In [ ]:
!mkdir dataset
%cd dataset
# !gdown # downloading mibench dataset
!tar -xvf programs.tar.gz

In [ ]:
import os
import shutil
from glob import glob
from pathlib import Path
from tqdm import tqdm
import subprocess

# Paths
SRC_ROOT = "/content/dataset/ProgramData"

DST_ROOT = "/content/tsvb_dataset_ll"

os.makedirs(DST_ROOT, exist_ok=True)

train = 1000
txt_files = glob(f"{SRC_ROOT}/**/*.txt", recursive=True)[:train]

print(f"Found {len(txt_files)} files. Starting compilation...")

for path in tqdm(txt_files):
    try:
        name = Path(path).stem
        c_path = f"/tmp/{name}.c"
        ll_path = f"{DST_ROOT}/{name}.ll"

        shutil.copy(path, c_path)
        subprocess.run(["clang", "-S", "-emit-llvm", c_path, "-o", ll_path],
                       check=True,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
    except Exception as e:
        print(f"❌ Failed to compile {path}: {e}")

print("\n✅ All valid .txt files converted to .ll and stored in:", DST_ROOT)

In [ ]:
!pip install transformers accelerate bitsandbytes
!pip install -U bitsandbytes

In [ ]:
from huggingface_hub import login

login("TOKEN")

In [ ]:
import os
from glob import glob
from pathlib import Path
from textwrap import indent
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAMES = ["facebook/llm-compiler-7b"]

# 4-bit quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

class LLM_Compiler:
    def __init__(self, model_name: str = "facebook/llm-compiler-7b", device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        if model_name not in MODEL_NAMES:
            raise ValueError(f"model_name must be one of {MODEL_NAMES}")
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
        self.model.eval()

    def infer(self, prompt: str, max_new_tokens: int = 50) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        text: str = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return text[len(prompt):]

    def optimize_for_code_size(self, ir: str, max_new_tokens: int = 50) -> str:
        prompt = f"""[INST] Tell me how to optimize this LLVM-IR for object file size:
<code>{ir}</code> [/INST]"""
        return self.infer(prompt, max_new_tokens=max_new_tokens)

    def suggest_passes_for_size(self, ir: str, max_new_tokens: int = 50) -> str:
        prompt = f"""[INST] Return only a comma-separated list of LLVM optimization passes to minimize code size:
<code>{ir}</code> [/INST]"""
        return self.infer(prompt, max_new_tokens=max_new_tokens)

if __name__ == "__main__":
    DST_ROOT = "/content/tsvb_dataset_ll"
    max_new_tokens = 4000

    llm_compiler = LLM_Compiler()

    ll_files = glob(f"{DST_ROOT}/*.ll")[:10]
    if not ll_files:
        print(f"No .ll files found in {DST_ROOT}. Run the data loading script first.")
        exit()

    print(f"\nProcessing {len(ll_files)} LLVM-IR samples from {DST_ROOT}\n")

    for ll_file in ll_files:
        with open(ll_file, 'r') as f:
            ir = f.read()
        if 'define' not in ir:
            print(f"Skipping {ll_file}: Invalid or empty LLVM-IR")
            continue
        if os.path.getsize(ll_file) > 10000:  # Skip files >10KB
            print(f"Skipping {ll_file}: File too large")
            continue

        file_name = Path(ll_file).name
        print(f"\nProcessing {file_name}:")

        # print("Suggested optimization passes:")
        # passes = llm_compiler.suggest_passes_for_siz\e(ir, max_new_tokens)
        # print(indent(passes, "    "))

        print("\nDetailed optimization suggestions:")
        detailed = llm_compiler.optimize_for_code_size(ir, max_new_tokens)
        print(indent(detailed, "    "))

Loading checkpoint shards: 100%
 3/3 [01:07<00:00, 21.81s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

Processing 50 LLVM-IR samples from /content/tsvb_dataset_ll

Skipping /content/tsvb_dataset_ll/2688.ll: File too large
Skipping /content/tsvb_dataset_ll/1449.ll: File too large
Skipping /content/tsvb_dataset_ll/2136.ll: File too large
Skipping /content/tsvb_dataset_ll/1952.ll: File too large
Skipping /content/tsvb_dataset_ll/1544.ll: File too large
Skipping /content/tsvb_dataset_ll/1135.ll: File too large

Processing 57.ll:

Detailed optimization suggestions:
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    <code>; ModuleID = '/tmp/57.c'
    source_filename = "/tmp/57.c"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-pc-linux-gnu"

    @.str = private unnamed_addr constant [3 x i8] c"%d\00", align 1
    @.str.1 = private unnamed_addr constant [7 x i8] c"%d%d%d\00", align 1
    @__const.main.month = private unnamed_addr constant [12 x i32] [i32 31, i32 29, i32 31, i32 30, i32 31, i32 30, i32 31, i32 31, i32 30, i32 31, i32 30, i32 31], align 16
    @__const.main.month.2 = private unnamed_addr constant [12 x i32] [i32 31, i32 28, i32 31, i32 30, i32 31, i32 30, i32 31, i32 31, i32 30, i32 31, i32 30, i32 31], align 16
    @.str.3 = private unnamed_addr constant [5 x i8] c"YES\0A\00", align 1
    @.str.4 = private unnamed_addr constant [4 x i8] c"NO\0A\00", align 1

    ; Function Attrs: noinline nounwind optnone uwtable
    define dso_local i32 @main() #0 {
      %1 = alloca i32, align 4
      %2 = alloca i32, align 4
      %3 = alloca i32, align 4
      %4 = alloca i32, align 4
      %5 = alloca i32, align 4
      %6 = alloca i32, align 4
      %7 = alloca i32, align 4
      %8 = alloca i32, align 4
      %9 = alloca i32, align 4
      %10 = alloca i32, align 4
      %11 = alloca i32, align 4
      %12 = alloca [12 x i32], align 16
      %13 = alloca i32, align 4
      %14 = alloca [12 x i32], align 16
      %15 = alloca i32, align 4
      store i32 0, i32* %1, align 4
      %16 = call i32 (i8*, ...) @scanf(i8* noundef getelementptr inbounds ([3 x i8], [3 x i8]* @.
Skipping /content/tsvb_dataset_ll/507.ll: File too large
Skipping /content/tsvb_dataset_ll/1466.ll: File too large
Skipping /content/tsvb_dataset_ll/2379.ll: File too large
Skipping /content/tsvb_dataset_ll/2029.ll: File too large
Skipping /content/tsvb_dataset_ll/3576.ll: File too large
Skipping /content/tsvb_dataset_ll/903.ll: File too large
Skipping /content/tsvb_dataset_ll/1435.ll: File too large
Skipping /content/tsvb_dataset_ll/1780.ll: File too large

Processing 842.ll:

Detailed optimization suggestions:
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    <code>; ModuleID = '/tmp/842.c'
    source_filename = "/tmp/842.c"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-pc-linux-gnu"

    @.str = private unnamed_addr constant [5 x i8] c"%d%d\00", align 1
    @.str.1 = private unnamed_addr constant [3 x i8] c"%d\00", align 1
    @.str.2 = private unnamed_addr constant [4 x i8] c"%d\0A\00", align 1

    ; Function Attrs: noinline nounwind optnone uwtable
    define dso_local i32 @main() #0 {
      %1 = alloca i32, align 4
      %2 = alloca i32, align 4
      %3 = alloca i32, align 4
      %4 = alloca [100 x [100 x i32]], align 16
      %5 = alloca i32, align 4
      %6 = alloca i32, align 4
      %7 = alloca i32, align 4
      %8 = alloca i32, align 4
      store i32 0, i32* %1, align 4
      %9 = call i32 (i8*, ...) @scanf(i8* noundef getelementptr inbounds ([5 x i8], [5 x i8]* @.str, i64 0, i64 0), i32* noundef %2, i32* noundef %3)
      store i32 0, i32* %5, align 4
      br label %10

    10:                                               ; preds = %31, %0
      %11 = load i32, i32* %5, align 4
      %12 = load i32, i32* %2, align 4
      %13 = icmp slt i32 %11, %12
      br i1 %13, label %14, label %34

    14:                                               ; preds = %10
      store i32 0, i32* %6, align 4
      br label %15

    15:                                               ; preds = %27, %14
      %16 = load i32, i32* %6, align 4
      %17 = load i32, i32* %3, align 4
      %18 = icmp slt i32 %16, %17
      br i1 %18, label %19, label %30

    19:                                               ; preds = %15
      %20 = load i32, i32* %5, align 4
      %21 = sext i32 %20 to i64
      %22 = getelementptr

Processing 544.ll:

Detailed optimization suggestions:
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
      The LLVM-IR will have instruction count 148 and binary sise 1007 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    @.str = private unnamed_addr constant [3 x i8] c"%d\00", align 1
    @.str.1 = private unnamed_addr constant [9 x i8] c"%d %d %d\00", align 1
    @__const.main.d = private unnamed_addr constant [12 x i32] [i32 31, i32 28, i32 31, i32 30, i32 31, i32 30, i32 31, i32 31, i32 30, i32 31, i32 30, i32 31], align 16
    @.str.2 = private unnamed_addr constant [5 x i8] c"YES\0A\00", align 1
    @.str.3 = private unnamed_addr constant [4 x i8] c"NO\0A\00", align 1

    ; Function Attrs: noinline nounwind optnone uwtable
    define dso_local i32 @main() #0 {
      %1 = alloca i32, align 4
      %2 = alloca i32, align 4
      %3 = alloca i32, align 4
      %4 = alloca i32, align 4
      %5 = alloca i8*, align 8
      %6 = alloca i64, align 8
      %7 = alloca i64, align 8
      %8 = alloca i64, align 8
      %9 = alloca [12 x i32], align 16
      %10 = alloca i32, align 4
      %11 = alloca i32, align 4
      %12 = alloca i32, align 4
      %13 = alloca i32, align 4
      %14 = alloca i32, align 4
      store i32 0, i32* %1, align 4
      %15 = call i32 (i8*, ...) @scanf(i8* noundef getelementptr inbounds ([3 x i8], [3 x i8]* @.str, i64 0, i64 0), i32* noundef %2)
      %16 = load i32, i32* %2, align 4
      %17 = zext i32 %16 to i64
      %18 = call i8* @llvm.stacksave()
      store i8* %18, i8** %5, align 8
      %19 = alloca i32, i64 %17, align
Skipping /content/tsvb_dataset_ll/757.ll: File too large
Skipping /content/tsvb_dataset_ll/2224.ll: File too large
Skipping /content/tsvb_dataset_ll/141.ll: File too large
Skipping /content/tsvb_dataset_ll/640.ll: File too large
Skipping /content/tsvb_dataset_ll/1969.ll: File too large

Processing 281.ll:

Detailed optimization suggestions:
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
      The LLVM-IR will have instruction count 148 and binary sise 616 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    @normal_months = dso_local global [12 x i32] [i32 31, i32 28, i32 31, i32 30, i32 31, i32 30, i32 31, i32 31, i32 30, i32 31, i32 30, i32 31], align 16
    @leap_months = dso_local global [12 x i32] [i32 31, i32 29, i32 31, i32 30, i32 31, i32 30, i32 31, i32 31, i32 30, i32 31, i32 30, i32 31], align 16
    @.str = private unnamed_addr constant [3 x i8] c"%d\00", align 1
    @.str.1 = private unnamed_addr constant [7 x i8] c"%d%d%d\00", align 1
    @.str.2 = private unnamed_addr constant [5 x i8] c"YES\0A\00", align 1
    @.str.3 = private unnamed_addr constant [4 x i8] c"NO\0A\00", align 1

    ; Function Attrs: noinline nounwind optnone uwtable
    define dso_local i32 @is_leap(i32 noundef %0) #0 {
      %2 = alloca i32, align 4
      %3 = alloca i32, align 4
      store i32 %0, i32* %3, align 4
      %4 = load i32, i32* %3, align 4
      %5 = srem i32 %4, 100
      %6 = icmp eq i32 %5, 0
      br i1 %6, label %7, label %12

    7:                                                ; preds = %1
      %8 = load i32, i32* %3, align 4
      %9 = srem i32 %8, 400
      %10 = icmp eq i32 %9, 0
      %11 = zext i1 %10 to i32
      store i32 %11, i32* %2, align 4
      br label %17

    12:                                               ; preds = %1
      %13 = load i32, i32* %3, align 4
      %14 = srem
Skipping /content/tsvb_dataset_ll/392.ll: File too large

Processing 79.ll:

Detailed optimization suggestions:
---------------------------------------------------------------------------
KeyboardInterrupt                         Traceback (most recent call last)
/tmp/ipython-input-7-2663672684.py in <cell line: 0>()
     83
     84         print("\nDetailed optimization suggestions:")
---> 85         detailed = llm_compiler.optimize_for_code_size(ir, max_new_tokens)
     86         print(indent(detailed, "    "))

18 frames
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py in _call_impl(self, *args, **kwargs)
   1748                 or _global_backward_pre_hooks or _global_backward_hooks
   1749                 or _global_forward_hooks or _global_forward_pre_hooks):
-> 1750             return forward_call(*args, **kwargs)
   1751
   1752         result = None

KeyboardInterrupt:
